# 04 · Performance Metrics — scoring a strategy
**Goal:** turn a stream of daily returns into the numbers that say whether a strategy is any good: the **equity curve**, **Sharpe**, **max drawdown**, **CAGR**, and **Sortino**.

> Maps to: project **08** (backtester) `backtester/metrics.py`. KB §08.

In [ ]:
%matplotlib inline
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
rng = np.random.default_rng(0)

## 1. From returns to an equity curve
Suppose a strategy produces these daily returns. Starting with \$1, the **equity curve** is the running product of `(1 + return)` — it shows your money growing (or shrinking) over time.

In [ ]:
days = 252 * 3
rets = pd.Series(rng.normal(0.0004, 0.01, days))  # small positive edge + noise
equity = (1 + rets).cumprod()
equity.plot(figsize=(9,3), title='Equity curve ($1 start)'); plt.ylabel('equity'); plt.show()

## 2. Sharpe ratio — return per unit of risk
The **Sharpe ratio** is the average return divided by volatility, annualized with √252. It's the classic 'is the reward worth the risk?' number. ~1 is decent; >2 from something simple is usually too good to be true.

In [ ]:
def sharpe(r, periods=252):
    sd = r.std()
    return np.sqrt(periods) * r.mean() / sd if sd > 0 else 0.0

print(f'annualized Sharpe: {sharpe(rets):.2f}')

## 3. Maximum drawdown — the worst pain
**Drawdown** is how far equity has fallen from its highest previous peak. The **maximum drawdown** is the worst such fall — 'how bad did it get?' Investors feel drawdown more than they feel average return.

In [ ]:
def max_drawdown(equity):
    peak = equity.cummax()
    dd = equity / peak - 1.0
    return dd.min(), dd

mdd, dd = max_drawdown(equity)
print(f'max drawdown: {mdd:.1%}')
dd.plot(figsize=(9,2.5), title='Drawdown', color='crimson'); plt.ylabel('drawdown'); plt.show()

## 4. CAGR and Sortino
**CAGR** = the smooth annual growth rate that gets you from start to end equity. **Sortino** = like Sharpe, but only penalizes *downside* volatility (you don't mind big up-days).

In [ ]:
def cagr(equity, periods=252):
    years = len(equity) / periods
    return equity.iloc[-1] ** (1/years) - 1

def sortino(r, periods=252):
    downside = r[r < 0].std()
    return np.sqrt(periods) * r.mean() / downside if downside > 0 else 0.0

print(f'CAGR:    {cagr(equity):.1%}')
print(f'Sortino: {sortino(rets):.2f}')

## 5. More examples: Sharpe can be fooled
A higher *average return* is not automatically better — risk matters. Compare two strategies: one with high return AND high volatility, one calmer.

In [ ]:
hi = pd.Series(rng.normal(0.0010, 0.030, days))   # high return, high risk
lo = pd.Series(rng.normal(0.0004, 0.008, days))   # lower return, low risk
for name, r in [('high-return/high-risk', hi), ('low-return/low-risk', lo)]:
    eq = (1+r).cumprod()
    print(f'{name:24s} CAGR {cagr(eq):6.1%}  Sharpe {sharpe(r):4.2f}  maxDD {max_drawdown(eq)[0]:6.1%}')

Often the *calmer* strategy has the better **Sharpe** and a far smaller drawdown, even with a lower headline return — because Sharpe rewards consistency. That's why pros quote risk-adjusted numbers, not raw returns.

### 🧪 Try it yourself
1. Set the mean return to 0 (`rng.normal(0.0, 0.01, days)`) — every metric collapses toward 'no edge'.
2. Annualize differently: change `periods=252` to `12` (monthly data) and see the Sharpe scaling change.
3. Make one giant down-day (`rets.iloc[100] = -0.25`) and watch max drawdown react while the average barely moves — drawdown captures pain that averages hide.

**You should see:** a modest Sharpe (~0.6), a real drawdown even with a positive edge, and a Sortino a bit higher than Sharpe. Try setting the mean return to 0 (`rng.normal(0.0, 0.01, days)`) and watch every metric collapse toward 'no edge'.

### In the project
These exact metrics — plus the **Probabilistic Sharpe Ratio** that flags overfitting — live in project **08** `backtester/metrics.py` and drive its tearsheet.